# 📸 텍스트 이미지 VQA — Qwen3-VL LoRA · 선택지 순열 TTA · 고해상도 재추론 · 앙상블 (Colab Pro+)

**핵심 설계 (0.70 → 0.95+ 목표)**
1. **모델 교체**: Qwen2.5-VL-3B(4bit) → **Qwen3-VL-8B-Instruct (bf16)**. 한국어 포함 32개 언어 OCR이 강화된 세대.
2. **분류형 학습/추론**: `generate → 문자열 파싱` 대신, 정답 위치의 **A/B/C/D 4개 토큰 logit만 뽑아 4-way CE로 학습**하고 같은 logit으로 추론 → 파싱 실패 0%, 학습·추론 목표 일치, 확률 기반 앙상블 가능. (`logits_to_keep=1`로 vocab 전체 logit 계산도 생략 → 메모리/속도 이득)
3. **선택지 셔플 증강 + 순열 TTA**: 학습 때 매 epoch 선택지 순서를 랜덤 셔플, 추론 때 4가지 순환 순열의 평균 → 위치/길이 편향 제거 (EDA: "가장 긴 선택지"만 골라도 35%).
4. **해상도**: 512px 고정 → 시각 토큰 768~1280(학습, 랜덤), 1600(추론), **확신도 낮은 샘플만 3200 토큰으로 재추론**. EXIF 회전 보정 + JPEG draft 디코딩.
5. **Colab 끊김 대비**: 체크포인트·예측 청크를 Drive에 저장하고 같은 셀을 다시 실행하면 이어서 진행.
6. **앙상블 + pseudo label**: run별 확률을 Drive에 저장 → 검증셋 기준 가중치 최적화 → 고신뢰 test 예측으로 재학습.

**권장 실행 순서** — 각 run은 설정 셀 값만 바꾸고 `0번 셀 → 세션 다시 시작 → 1번 셀부터 셀 및 하위 셀 실행`
| 순서 | 설정 | 하는 일 | G4 예상 |
|---|---|---|---|
| A | `qwen3vl_32b` · `holdout` · `USE_DEV=False` | 학습된 run 재실행 → dev 예측만 계산 → `dev_labels.csv` 생성 | ~1h |
| T | `RUN_MODE="tts"` (+ 0번 셀 flash-linear-attention 주석 해제) | 앙상블이 헷갈리는 문항만 Qwen3.8-27B thinking으로 재판단, 5-fold 교차검증으로 채택 결정 → `submission_final.csv` | ~2~2.5h |
| B | `qwen3vl_32b` · `holdout` · `USE_DEV=True` | train + dev 라벨로 학습 → 검증 점수로 dev 효과 측정 | ~9~10h |
| T' | `RUN_MODE="tts"` 다시 | 앙상블이 바뀐 만큼 새 문항만 추가 재판단 (나머지는 캐시) → `submission_final.csv` | ~0.5h |
| C | B가 좋으면 `full` · `USE_DEV=True` | 최종 학습 → 앙상블 → T 다시 | ~8h |

**제출 파일**: T를 실행했다면 `submission_final.csv`, 아니면 `submission_ensemble.csv`
⚠️ `USE_PSEUDO`(test 예측을 학습에 사용)는 규칙상 위험하므로 계속 False. dev는 대회가 증강용으로 명시 허용한 데이터입니다.

**규칙 준수**: HF 공개 모델만 사용, API 추론 없음, 모든 튜닝(에폭 선택·고해상도 임계값·앙상블 가중치)은 train에서 분리한 검증셋으로만 수행, test는 추론에만 사용.
**재현성**: 패키지 버전 고정 + 결정론 설정 + run별 환경 정보(`runs/<RUN_NAME>/env.json`, 모델 commit hash 포함) 기록. 제출 시 `runs/*/best|last`(LoRA 어댑터)와 `preds/`를 함께 제출하면 추론만으로 동일한 CSV를 재현할 수 있습니다. (GPU bf16 연산 특성상 *재학습*은 bit 단위로 동일하지 않을 수 있음)

**데이터 준비**: 받은 zip·csv 파일을 압축 풀지 말고 `MyDrive/vqa_challenge/` 폴더에 그대로 올리면 됩니다
(파일 이름·zip 개수 무관, zip 안의 zip·상위 폴더도 자동 처리).

In [ ]:
# [colab-only] 0. GPU 확인 & 패키지 설치 → 설치 후 '세션 다시 시작' 필수
!nvidia-smi --query-gpu=name,memory.total --format=csv
%pip -q install "transformers==5.17.0" "peft==0.21.0" "accelerate==1.15.0" "safetensors==0.8.0" "bitsandbytes>=0.46.0"
%pip -q uninstall -y torchao   # Colab 기본 torchao(구버전)가 peft와 충돌 → 이 노트북은 torchao를 쓰지 않으므로 제거
# ⚠️ Pillow/pandas/tqdm는 Colab 기본 버전을 그대로 씁니다 (실행 중 업그레이드하면 PIL ImportError 발생).
# ⚠️ 이 셀을 처음 실행한 뒤에는 반드시 [런타임 → 세션 다시 시작] 후 다음 셀부터 실행하세요.
# Qwen3.8 / Qwen3.5 계열을 쓸 때만 아래 주석 해제 (Gated DeltaNet 고속 커널, 없으면 수 배 느림)
# %pip -q install flash-linear-attention

name, memory.total [MiB]
NVIDIA RTX PRO 6000 Blackwell Server Edition, 97887 MiB
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 197.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 83.6 MB/s eta 0:00:00


In [ ]:
# 1. 설정 ────────────────────────────────────────────────────────────────
import os, gc, re, json, math, time, glob, shutil, random, zipfile, inspect, hashlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")      # 재현성: cuBLAS 결정론 (CUDA 초기화 전에 설정)
warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── 경로 ──
DRIVE_ROOT = Path("/content/drive/MyDrive/vqa_challenge")   # 체크포인트·예측·제출 파일 저장 (세션이 끊겨도 유지)
DATA_DIR   = DRIVE_ROOT / "data"                             # 이 폴더가 없으면 DRIVE_ROOT 바로 아래의 zip/csv를 사용
LOCAL_DATA = Path("/content/data")                           # 로컬 SSD로 복사해서 사용 (Drive 직접 읽기는 매우 느림)

# ── 실행 모드 ──
MODEL_KEY  = "qwen3vl_32b"  # qwen3vl_8b | qwen3vl_32b | qwen3_8_27b(최신 세대) | qwen3vl_4b
RUN_MODE   = "full"         # holdout: 90% 학습 + 10% 검증 | full: 100% 학습 | zeroshot: 학습 없이 추론
                            # tts: 앙상블이 헷갈리는 문항만 thinking 모델로 재판단 (학습 없음, 13번 셀)
USE_PSEUDO = False          # True: 앙상블 셀이 만든 pseudo_labels.csv(고신뢰 test 예측)를 학습에 추가
USE_DEV    = False          # True: dev.csv(교육생 응답 5개) + 모델 예측으로 만든 dev_labels.csv를 학습에 추가
PREDICT_DEV = False         # True: 추론 셀에서 dev 예측도 계산 (dev 라벨을 새로 만들 때만 필요, 32B 기준 약 50분)
SMOKE_TEST = False          # True: 소량 데이터로 전체 파이프라인만 빠르게 점검
SEED       = 42             # 학습 시드 (시드만 바꿔 여러 번 학습 → 앙상블 멤버로 사용 가능)
RUN_TAG    = ""             # 같은 모델의 다른 실험을 별도 앙상블 멤버로 두고 싶을 때 (예: "vis", "r64")
SPLIT_SEED = 42             # 검증 분할은 모든 run에서 고정 (앙상블 가중치 튜닝의 전제) — 바꾸지 마세요

# ── 학습 ──
EPOCHS          = 2
LR              = 1e-4
WARMUP_RATIO    = 0.03
LORA_R          = 32
LORA_DROPOUT    = 0.05
LORA_VISION     = False     # True: 비전 인코더에도 LoRA (한글 OCR 적응, 메모리↑) — RUN_TAG="vis"와 함께 실험 권장
LABEL_SMOOTHING = 0.0      # 0.05면 확신도가 0.96에 묶여 정답/오답 구분력이 사라짐 (오류 분석 결과)
VALID_RATIO     = 0.10
SAVE_EVERY      = 50        # optimizer step마다 Drive에 체크포인트 (끊기면 여기서 재개)

# ── 해상도: '시각 토큰 수' 기준 (Qwen3-VL은 1토큰 = 32x32px) ──
TRAIN_TOKENS = (768, 1280)  # 학습 시 샘플마다 랜덤 → 해상도 증강 + 속도
INFER_TOKENS = 1600
HIRES_TOKENS = 3200         # 확신도 낮은 샘플만 이 해상도로 재추론
MIN_TOKENS   = 256

# ── 추론 ──
N_TTA            = 4        # 1 | 2 | 4 | 8  (선택지 순서 순환 TTA)
HIRES_GRID       = (0.6, 0.8, 0.9, 0.95, 0.99)   # 고해상도 재추론 확신도 임계값 후보 → 검증셋에서 자동 선택
HARD_QTYPES      = ("가격", "방향", "개수", "전화번호")  # 확신도와 무관하게 고해상도도 볼지 검증셋으로 결정
PSEUDO_THRESHOLD = 0.97     # 앙상블 확신도가 이 이상인 test 예측만 pseudo label로 사용 (dev 라벨의 확신 기준으로도 사용)
DEV_TRUST        = 0.95     # 투표 패턴별로 '모델 확신 문항과 다수결 일치율'이 이 이상이면 그 패턴의 다수결을 라벨로 신뢰
QUICK_CHECK_N    = 200      # 학습 전 zero-shot 정확도 빠른 측정 샘플 수 (0이면 생략)
MODEL_REVISION   = "main"   # 재현성: 첫 실행 후 env.json의 model_commit 값으로 바꿔 고정하세요

# ── 선택적 추론 재판단 (RUN_MODE="tts") ──
TTS_MODEL_KEY  = "qwen3_8_27b"   # thinking 모드를 지원하는 최신 모델
TTS_CONF_MAX   = 0.99            # 앙상블 확신도가 이 값 미만인 문항만 후보 (실제 임계값은 검증셋 교차검증으로 선택)
TTS_MAX_TOKENS = 1536            # 추론 길이 상한 (넘으면 강제로 추론을 닫고 답을 채점)
TTS_BS         = 16              # 메모리 부족 시 자동으로 절반씩 나눠 재시도

MODEL_REGISTRY = {
    "qwen3vl_4b":  {"id": "Qwen/Qwen3-VL-4B-Instruct",  "params_b": 4.4},
    "qwen3vl_8b":  {"id": "Qwen/Qwen3-VL-8B-Instruct",  "params_b": 8.8},
    "qwen3vl_32b": {"id": "Qwen/Qwen3-VL-32B-Instruct", "params_b": 33.4},
    # ↓ Qwen3.5 구조(Gated DeltaNet 하이브리드) 계열: 0번 셀의 flash-linear-attention 설치 줄 주석 해제 필수
    "qwen3_8_27b": {"id": "Qwen/Qwen3.8-27B",           "params_b": 27.8},   # 2026.08 최신 세대, Apache-2.0
    "qwen3_5_9b":  {"id": "Qwen/Qwen3.5-9B",            "params_b": 9.7},
    "qwen3_6_27b": {"id": "Qwen/Qwen3.6-27B",           "params_b": 27.8},
}

PROMPT_VER = "p3"            # 프롬프트를 바꾸면 이 값도 바꿀 것 (이전 어댑터·캐시와 섞이지 않도록)
TAG = PROMPT_VER + (f"_{RUN_TAG}" if RUN_TAG else "")
if RUN_MODE == "tts":
    MODEL_KEY = TTS_MODEL_KEY
RUN_NAME = f"{MODEL_KEY}_{RUN_MODE}_s{SEED}_{TAG}" if RUN_MODE != "tts" else f"tts_{MODEL_KEY}_{TAG}"
if USE_PSEUDO and RUN_MODE != "zeroshot":
    RUN_NAME += "_pl"
if USE_DEV and RUN_MODE != "zeroshot":
    RUN_NAME += "_dev"
if SMOKE_TEST:
    RUN_NAME += "_smoke"
    EPOCHS, SAVE_EVERY, QUICK_CHECK_N = 1, 2, 8
RUN_DIR  = DRIVE_ROOT / "runs" / RUN_NAME
PRED_DIR = DRIVE_ROOT / "preds"
for d in (RUN_DIR, PRED_DIR):
    d.mkdir(parents=True, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = True, False
torch.use_deterministic_algorithms(True, warn_only=True)   # 결정론 커널 우선 (없는 연산은 경고만)
print("RUN_NAME:", RUN_NAME)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RUN_NAME: qwen3vl_32b_full_s42_p3


In [ ]:
# 2. 하드웨어 자동 설정 ──────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    _p = torch.cuda.get_device_properties(0)
    GPU_NAME, VRAM_GB = _p.name, _p.total_memory / 1024**3
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    assert torch.cuda.is_bf16_supported(), "bf16 미지원 GPU(T4 등)입니다. 런타임 유형을 A100/L4/H100/G4로 바꾸세요."
else:
    GPU_NAME, VRAM_GB = "cpu", 0.0

_spec = MODEL_REGISTRY[MODEL_KEY]
_need_gb = _spec["params_b"] * 2.05 + (14 if RUN_MODE in ("holdout", "full") else 6)
USE_4BIT = DEVICE == "cuda" and _need_gb > VRAM_GB * 0.95        # 부족할 때만 QLoRA(4bit)로 자동 전환

if VRAM_GB >= 70:
    TRAIN_BS, INFER_BS = 4, 4
elif VRAM_GB >= 35:
    TRAIN_BS, INFER_BS = 2, 2
else:
    TRAIN_BS, INFER_BS = 1, 1
if _spec["params_b"] > 20:
    TRAIN_BS, INFER_BS = max(1, TRAIN_BS // 2), max(1, INFER_BS // 2)
GRAD_ACCUM  = max(1, 16 // TRAIN_BS)                                 # 유효 배치 16 고정
GRAD_CKPT   = True     # 80GB↑ GPU + 8B면 False로 ~30% 빨라질 수 있음 (OOM 나면 True)
NUM_WORKERS = min(8, max(2, (os.cpu_count() or 4) - 2))
PATCH_FACTOR = 32                                                    # processor 로드 후 실제 값으로 갱신

print(f"GPU={GPU_NAME} ({VRAM_GB:.0f}GB) | 4bit={USE_4BIT} | train_bs={TRAIN_BS}x{GRAD_ACCUM} | "
      f"infer_bs={INFER_BS}x{N_TTA}perm | workers={NUM_WORKERS}")

GPU=NVIDIA RTX PRO 6000 Blackwell Server Edition (95GB) | 4bit=False | train_bs=2x8 | infer_bs=2x4perm | workers=8


In [ ]:
# 3. 데이터 준비 · 검증 분할 ─────────────────────────────────────────────
IMG_EXT  = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
OPT_COLS = ["a", "b", "c", "d"]

def prepare_local_data():
    """Drive → 로컬 SSD 복사 + 압축 해제. 지원하는 배치:
       ① vqa_challenge/data/ 폴더   ② vqa_challenge/ 바로 아래의 zip·csv 파일들 (data.zip 하나든, train.zip·test.zip·csv든 이름 무관)"""
    need = ["train.csv", "test.csv", "sample_submission.csv"]
    if all((LOCAL_DATA / n).exists() for n in need):
        return
    LOCAL_DATA.mkdir(parents=True, exist_ok=True)
    if DATA_DIR.exists():
        print("Drive 폴더 → 로컬 복사:", DATA_DIR)
        shutil.copytree(DATA_DIR, LOCAL_DATA, dirs_exist_ok=True)
    else:
        skip = ("split_", "pseudo_", "submission")                     # 이 노트북이 만든 결과 파일은 제외
        srcs = [f for f in sorted(DRIVE_ROOT.iterdir())
                if f.is_file() and f.suffix.lower() in (".zip", ".csv") and not f.name.startswith(skip)]
        if not srcs:
            raise FileNotFoundError(f"{DRIVE_ROOT} 에 zip/csv 파일이 없습니다. 데이터 파일을 이 폴더에 올려주세요.")
        for f in srcs:
            print(f"Drive → 로컬 복사: {f.name} ({f.stat().st_size / 1024**2:.0f}MB)")
            shutil.copy(f, LOCAL_DATA / f.name)
    for _ in range(2):                                                   # zip 안의 zip까지 (최대 2단계)
        for z in list(LOCAL_DATA.rglob("*.zip")):
            if "__MACOSX" in z.parts:
                continue
            print("압축 해제:", z.name)
            with zipfile.ZipFile(z) as zf:
                zf.extractall(z.parent)
            z.unlink()
    for n in need:                                                       # csv가 하위 폴더에 있으면 최상위로
        if not (LOCAL_DATA / n).exists():
            found = [p for p in LOCAL_DATA.rglob(n) if "__MACOSX" not in p.parts]
            if not found:
                raise FileNotFoundError(f"{n} 을(를) 찾지 못했습니다. 업로드한 파일을 확인하세요.")
            shutil.copy(found[0], LOCAL_DATA / n)
    if not (LOCAL_DATA / "dev.csv").exists():                          # dev.csv는 선택
        found = [p for p in LOCAL_DATA.rglob("dev.csv") if "__MACOSX" not in p.parts]
        if found:
            shutil.copy(found[0], LOCAL_DATA / "dev.csv")
    n_img = sum(1 for p in LOCAL_DATA.rglob("*") if p.suffix.lower() in IMG_EXT and "__MACOSX" not in p.parts)
    print(f"로컬 데이터 준비 완료: 이미지 {n_img}개")

def clean_text(s):
    s = str(s).replace("\r\n", "\n").replace("\r", "\n")
    s = " / ".join(t.strip() for t in s.split("\n") if t.strip())    # 여러 줄 선택지 → 한 줄
    return re.sub(r"[ \t]+", " ", s).strip()

QTYPE_RULES = [
    ("방향",     ["방향", "어디로", "오른쪽", "왼쪽", "향하", "화살표"]),
    ("전화번호", ["전화", "번호", "연락처"]),
    ("가격",     ["가격", "얼마", "요금", "비용", "금액"]),
    ("날짜시간", ["날짜", "시간", "요일", "영업", "기간", "몇 시", "언제"]),
    ("색상",     ["색깔", "색상", "무슨 색", "어떤 색"]),
    ("개수",     ["몇 개", "몇 명", "개수", "몇 층"]),
    ("명칭",     ["이름", "상호", "제목", "명칭", "브랜드"]),
]
def qtype_of(q):
    for name, kws in QTYPE_RULES:
        if any(k in q for k in kws):
            return name
    return "문구/기타"

prepare_local_data()
PATH_INDEX = {p.name: str(p) for p in LOCAL_DATA.rglob("*")
              if p.suffix.lower() in IMG_EXT and "__MACOSX" not in p.parts}

def load_csv(name):
    df = pd.read_csv(LOCAL_DATA / name)
    for c in ["question"] + OPT_COLS:
        df[c] = df[c].map(clean_text)
    df["abs_path"] = [str(LOCAL_DATA / p) if (LOCAL_DATA / p).exists() else PATH_INDEX.get(Path(p).name)
                      for p in df["path"]]
    miss = df["abs_path"].isna().sum()
    assert miss == 0, f"{name}: 이미지 {miss}개를 찾지 못함 → 예: {df.loc[df.abs_path.isna(), 'path'].head(3).tolist()}"
    df["qtype"] = df["question"].map(qtype_of)
    return df

train_all = load_csv("train.csv")
train_all["answer"] = train_all["answer"].astype(str).str.strip().str.lower()
assert set(train_all["answer"]) <= set(OPT_COLS)
test_df = load_csv("test.csv")
sample_sub = pd.read_csv(LOCAL_DATA / "sample_submission.csv")

# dev: 정답 없음, 교육생 응답 answer1~5. test와 질문+선지가 완전히 같은 문항은 누수 방지를 위해 제외
DEV_COLS = [f"answer{i}" for i in range(1, 6)]
dev_df = pd.DataFrame()
if (LOCAL_DATA / "dev.csv").exists():
    try:
        dev_df = load_csv("dev.csv")
        _k = lambda d: d["question"].str.lower() + "||" + d[OPT_COLS].apply(
            lambda r: "|".join(sorted(str(x).lower() for x in r)), axis=1)
        _dup = _k(dev_df).isin(set(_k(test_df)))
        dev_df = dev_df[~_dup].reset_index(drop=True)
        print(f"dev {len(dev_df)}개 로드 (test와 동일 문항 {int(_dup.sum())}개 제외)")
    except AssertionError as e:
        print("⚠️ dev 이미지를 찾지 못해 dev를 사용하지 않습니다:", str(e)[:120])
        dev_df = pd.DataFrame()

# 모든 run이 동일한 검증셋을 쓰도록 분할을 Drive에 고정 저장 (앙상블 가중치 튜닝의 전제)
SPLIT_CSV = DRIVE_ROOT / f"split_s{SPLIT_SEED}.csv"
if SPLIT_CSV.exists():
    split = pd.read_csv(SPLIT_CSV)
else:
    rng = np.random.default_rng(SPLIT_SEED)
    strata = train_all["answer"] + "_" + train_all["qtype"]
    is_valid = np.zeros(len(train_all), dtype=bool)
    for _, idx in train_all.groupby(strata).groups.items():
        idx = np.array(list(idx))
        if len(idx) < 2:
            continue
        n_val = max(1, int(round(len(idx) * VALID_RATIO)))
        is_valid[rng.choice(idx, n_val, replace=False)] = True
    split = pd.DataFrame({"id": train_all["id"], "is_valid": is_valid})
    split.to_csv(SPLIT_CSV, index=False)
train_all = train_all.merge(split, on="id", how="left")
valid_df = train_all[train_all["is_valid"]].reset_index(drop=True)
train_fit_df = (train_all if RUN_MODE == "full" else train_all[~train_all["is_valid"]]).reset_index(drop=True)

PSEUDO_CSV = DRIVE_ROOT / "pseudo_labels.csv"
if USE_PSEUDO and RUN_MODE != "zeroshot":
    assert PSEUDO_CSV.exists(), "pseudo_labels.csv가 없습니다. 먼저 앙상블 셀을 실행하세요."
    pl = pd.read_csv(PSEUDO_CSV)[["id", "answer"]]
    pl_df = test_df.merge(pl, on="id", how="inner")
    train_fit_df = pd.concat([train_fit_df, pl_df], ignore_index=True)
    print(f"pseudo label {len(pl_df)}개 추가")

DEV_LABEL_CSV = DRIVE_ROOT / "dev_labels.csv"
if USE_DEV and RUN_MODE != "zeroshot":
    assert DEV_LABEL_CSV.exists(), "dev_labels.csv가 없습니다. dev 예측이 있는 run 후 앙상블 셀을 먼저 실행하세요."
    _dl = pd.read_csv(DEV_LABEL_CSV)[["id", "answer"]]
    _dl_df = dev_df.drop(columns=DEV_COLS, errors="ignore").merge(_dl, on="id", how="inner")
    train_fit_df = pd.concat([train_fit_df, _dl_df], ignore_index=True)
    print(f"dev 라벨 {len(_dl_df)}개 추가")

if SMOKE_TEST:
    train_fit_df, valid_df, test_df = train_fit_df.head(48), valid_df.head(16), test_df.head(16)
    dev_df = dev_df.head(16)

print(f"train_fit={len(train_fit_df)} | valid={len(valid_df)} | test={len(test_df)}")
print(valid_df["qtype"].value_counts().to_dict())

Drive → 로컬 복사: data.zip (1716MB)
Drive → 로컬 복사: dev.csv (1MB)
Drive → 로컬 복사: dev.zip (294MB)
Drive → 로컬 복사: dev_labels.csv (0MB)
압축 해제: data.zip
압축 해제: dev.zip
로컬 데이터 준비 완료: 이미지 18794개
dev 2682개 로드 (test와 동일 문항 1개 제외)
train_fit=6714 | valid=670 | test=6714
{'문구/기타': 304, '명칭': 181, '가격': 88, '전화번호': 35, '날짜시간': 33, '방향': 18, '개수': 7, '색상': 4}


In [ ]:
# 4. 이미지 · 프롬프트 유틸 ─────────────────────────────────────────────
from PIL import Image, ImageOps, ImageFile
Image.MAX_IMAGE_PIXELS = None
ImageFile.LOAD_TRUNCATED_IMAGES = True
LETTERS = ["A", "B", "C", "D"]

def load_image(path, max_tokens, min_tokens=MIN_TOKENS):
    """시각 토큰 예산에 맞춰 리사이즈. EXIF 회전 보정 + JPEG draft 디코딩(대용량 사진 로딩 3~5배 빠름)."""
    max_px, min_px = max_tokens * PATCH_FACTOR ** 2, min_tokens * PATCH_FACTOR ** 2
    img = Image.open(path)
    w, h = img.size
    if w * h > max_px:
        s = math.sqrt(max_px / (w * h))
        try:
            img.draft("RGB", (max(1, int(w * s)), max(1, int(h * s))))
        except Exception:
            pass
    img = ImageOps.exif_transpose(img).convert("RGB")    # 휴대폰 사진이 옆으로 누운 채 들어가는 것 방지
    w, h = img.size
    if max(w, h) > 100 * min(w, h):                      # 극단적 종횡비 보호 (processor 한계 200:1)
        side = math.ceil(max(w, h) / 100)
        canvas = Image.new("RGB", (max(w, side), max(h, side)), (255, 255, 255))
        canvas.paste(img, ((canvas.width - w) // 2, (canvas.height - h) // 2))
        img, (w, h) = canvas, canvas.size
    area = w * h
    s = math.sqrt(max_px / area) if area > max_px else (math.sqrt(min_px / area) if area < min_px else 1.0)
    if abs(s - 1.0) > 1e-3:
        img = img.resize((max(PATCH_FACTOR, round(w * s)), max(PATCH_FACTOR, round(h * s))),
                         Image.Resampling.BICUBIC)
    return img

SYSTEM_PROMPT = (
    "You are an expert at reading text in real-world photos: Korean/English signboards, menus, posters, "
    "price tags, product labels, notices, maps and screens. Read every relevant character carefully, "
    "then answer the multiple-choice question about the image.\n"
    "The answer key of this dataset follows these conventions, and they take priority over what seems "
    "semantically most correct:\n"
    "1. Prominence: the largest, most eye-catching text wins. For a brand/store/product-name question, the big "
    "main name beats a small maker name or logo; for a price question, the big displayed price is usually the "
    "answer; for 'what does this sign mean', describe the main large sign, not a small sub-sign.\n"
    "2. Position / reading order: among similar candidates, the one at the top, center, or read first wins "
    "(the first line of an inscription, the top-left item of a notice, the main building rather than a "
    "neighboring one).\n"
    "3. Script: if the same name is written in several scripts (Kanji/Kana and Roman letters), the Latin/English "
    "spelling wins.\n"
    "Options may differ slightly from the image text (typos, spacing); choose the closest match. "
    "Watch for negations such as '않는', '아닌', '없는'."
)

def build_user_text(question, options):
    lines = [f"Question: {question}", "Options:"] + [f"{L}. {o}" for L, o in zip(LETTERS, options)]
    lines.append("Answer with the option's letter from the given choices directly.")
    return "\n".join(lines)

CHAT_KWARGS = {}
def render_prompt(question, options):
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [{"type": "image"},
                                     {"type": "text", "text": build_user_text(question, options)}]},
    ]
    return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, **CHAT_KWARGS)

def get_perms(n):
    """perm[j] = 화면의 j번째 자리(A~D)에 놓일 원래 선택지 인덱스. 순환 순열이라 각 선택지가 모든 자리를 한 번씩 거침."""
    cyc = [[(j + k) % 4 for j in range(4)] for k in range(4)]
    rev = [[(k - j) % 4 for j in range(4)] for k in range(4)]
    return {1: cyc[:1], 2: [cyc[0], cyc[2]], 4: cyc, 8: cyc + rev}[n]

def log_softmax_np(x):
    x = x - x.max(1, keepdims=True)
    return x - np.log(np.exp(x).sum(1, keepdims=True))

def acc_of(lp, df):
    return float((lp.argmax(1) == df["answer"].map(OPT_COLS.index).values).mean())

In [ ]:
# 5. 모델 로딩 함수 ──────────────────────────────────────────────────────
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, set_peft_model_state_dict
from safetensors.torch import load_file as load_safetensors

# 언어모델의 모든 선형층(비전 인코더 제외). Qwen3.5의 Gated-DeltaNet 투영층(in_proj_*, out_proj)도 포함.
LORA_TARGET_REGEX = (r"^(?!.*(?:visual|vision)).*\."
                     r"(?:q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj|in_proj\w*|out_proj)$")
if LORA_VISION:   # 비전 블록의 attention/MLP 선형층 추가 (merger·deepstack merger 제외)
    LORA_TARGET_REGEX = "(?:" + LORA_TARGET_REGEX + r")|(?:^.*visual\.blocks\.\d+\..*\.(?:qkv|proj|linear_fc1|linear_fc2)$)"

def setup_processor(model_id):
    global processor, PATCH_FACTOR, CHAT_KWARGS, LETTER_IDS
    processor = AutoProcessor.from_pretrained(model_id, revision=MODEL_REVISION)
    processor.tokenizer.padding_side = "left"              # 마지막 위치 = 정답 토큰 위치가 되도록
    ip = processor.image_processor
    PATCH_FACTOR = int(getattr(ip, "patch_size", 16) * getattr(ip, "merge_size", 2))
    lo, hi = 16 * PATCH_FACTOR ** 2, 16384 * PATCH_FACTOR ** 2   # 해상도는 load_image가 통제 → processor는 재축소 금지
    ip.size = {"shortest_edge": lo, "longest_edge": hi}
    for k, v in (("min_pixels", lo), ("max_pixels", hi)):
        if hasattr(ip, k):
            setattr(ip, k, v)
    # Qwen3.5 등 thinking 모델만: 생각 블록 없이 바로 답. (템플릿이 쓰지 않는 변수를 넘기면 transformers가 매 호출 경고를 띄움)
    tmpl = processor.chat_template if isinstance(processor.chat_template, str) else json.dumps(processor.chat_template)
    CHAT_KWARGS = {"enable_thinking": False} if "enable_thinking" in (tmpl or "") else {}
    LETTER_IDS = []
    for L in LETTERS:
        ids = processor.tokenizer.encode(L, add_special_tokens=False)
        assert len(ids) == 1, f"'{L}' 가 단일 토큰이 아님: {ids}"
        LETTER_IDS.append(ids[0])
    return processor

def load_base_model(model_id):
    kw = dict(dtype=torch.bfloat16, attn_implementation="sdpa", revision=MODEL_REVISION)
    if DEVICE == "cuda":
        kw["device_map"] = {"": 0}
    if USE_4BIT:
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            llm_int8_skip_modules=["visual", "lm_head"])    # 비전 인코더는 bf16 유지 (OCR 품질)
    try:
        m = AutoModelForImageTextToText.from_pretrained(model_id, **kw)
    except TypeError:                                        # 구버전 transformers
        kw["torch_dtype"] = kw.pop("dtype")
        m = AutoModelForImageTextToText.from_pretrained(model_id, **kw)
    m.config.use_cache = False
    return m

def attach_lora(m):
    if GRAD_CKPT:
        m.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    m.enable_input_require_grads()
    cfg = LoraConfig(r=LORA_R, lora_alpha=2 * LORA_R, lora_dropout=LORA_DROPOUT, bias="none",
                     target_modules=LORA_TARGET_REGEX, task_type="CAUSAL_LM")
    m = get_peft_model(m, cfg)
    m.print_trainable_parameters()
    return m

In [ ]:
# 6. Dataset · 스코어링 · 추론 함수 ──────────────────────────────────────
from torch.utils.data import Dataset, DataLoader

class VQADataset(Dataset):
    def __init__(self, df, train, perms=None, tokens=INFER_TOKENS, epoch=0):
        self.rows, self.train, self.epoch = df.to_dict("records"), train, epoch
        self.perms, self.tokens = perms or [[0, 1, 2, 3]], tokens

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        opts = [r[c] for c in OPT_COLS]
        if self.train:
            rng = np.random.default_rng([SEED, self.epoch, i])   # 재개해도 같은 증강이 나오도록 결정적 시드
            perm = rng.permutation(4).tolist()
            tok = int(rng.integers(TRAIN_TOKENS[0], TRAIN_TOKENS[1] + 1))
            img = load_image(r["abs_path"], tok)
            gold = OPT_COLS.index(r["answer"])
            return {"image": img, "texts": [render_prompt(r["question"], [opts[j] for j in perm])],
                    "target": perm.index(gold), "idx": i}
        img = load_image(r["abs_path"], self.tokens)
        return {"image": img, "idx": i,
                "texts": [render_prompt(r["question"], [opts[j] for j in p]) for p in self.perms]}

class Collator:
    def __call__(self, items):
        texts, images = [], []
        for it in items:
            texts += it["texts"]
            images += [it["image"]] * len(it["texts"])
        enc = processor(text=texts, images=images, padding=True, return_tensors="pt")
        batch = {"inputs": dict(enc), "idx": torch.tensor([it["idx"] for it in items])}
        if "target" in items[0]:
            batch["target"] = torch.tensor([it["target"] for it in items])
        return batch

def _worker_init(_):
    torch.set_num_threads(1)

class _ForkSafeLoader:
    """워커 프로세스가 부모의 이전 DataLoader 객체를 GC하며 내는 경고를 막기 위해, fork 순간에만 gc.freeze()"""
    def __init__(self, loader):
        self.loader = loader
    def __len__(self):
        return len(self.loader)
    def __iter__(self):
        gc.collect()
        gc.freeze()
        try:
            it = iter(self.loader)        # 이 시점에 워커가 fork됨
        finally:
            gc.unfreeze()
        return it

def make_loader(ds, bs, sampler=None):
    kw = dict(batch_size=bs, collate_fn=Collator(), num_workers=NUM_WORKERS,
              pin_memory=(DEVICE == "cuda"), worker_init_fn=_worker_init)
    if NUM_WORKERS > 0:
        kw["prefetch_factor"] = 4
    if sampler is not None:
        kw["sampler"] = sampler
    return _ForkSafeLoader(DataLoader(ds, **kw))

def letter_logits(m, inputs):
    """정답 위치(마지막 토큰)의 A/B/C/D logit만 반환 → [N, 4]"""
    inputs = {k: (v.to(DEVICE, non_blocking=True) if torch.is_tensor(v) else v) for k, v in inputs.items()}
    extra = {"logits_to_keep": 1} if SUPPORTS_LTK else {}
    with torch.autocast(device_type=DEVICE, dtype=torch.bfloat16):
        out = m(**inputs, use_cache=False, **extra)
    return out.logits[:, -1, LETTER_IDS].float()

def unpermute(lp, perms):
    """lp[B,P,4]: 표시 순서 기준 log-prob → 원래 선택지 순서로 되돌린 뒤 순열 평균 → [B,4]"""
    B, P, _ = lp.shape
    idx = torch.tensor(perms).unsqueeze(0).expand(B, P, 4)
    orig = torch.empty_like(lp).scatter_(2, idx, lp)
    return orig.mean(1).log_softmax(-1)

def _score_one_by_one(m, ds, i, perms):                    # OOM 시 fallback: 순열을 하나씩
    it = ds[i]
    lps = []
    for t in it["texts"]:
        enc = processor(text=[t], images=[it["image"]], padding=True, return_tensors="pt")
        lps.append(letter_logits(m, dict(enc)).log_softmax(-1).cpu())
    return unpermute(torch.stack(lps, 1), perms)

@torch.no_grad()
def predict_logprobs(m, df, n_tta=N_TTA, tokens=INFER_TOKENS, bs=None, desc="infer"):
    perms = get_perms(n_tta)
    P = len(perms)
    ds = VQADataset(df, train=False, perms=perms, tokens=tokens)
    out = np.full((len(df), 4), np.nan, dtype=np.float32)
    was_training = m.training
    m.eval()
    for batch in tqdm(make_loader(ds, bs or INFER_BS), desc=desc, leave=False):
        idx = batch["idx"]
        try:
            lg = letter_logits(m, batch["inputs"])
            lp = unpermute(lg.log_softmax(-1).view(-1, P, 4).cpu(), perms)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            lp = torch.cat([_score_one_by_one(m, ds, int(i), perms) for i in idx])
        out[idx.numpy()] = lp.numpy()
    if was_training:
        m.train()
    assert not np.isnan(out).any()
    return out

MODEL_FINGERPRINT = "base"
def predict_cached(m, df, tag, n_tta=N_TTA, tokens=INFER_TOKENS, bs=None, chunk=1000):
    """1000개 단위로 Drive에 캐시 → 추론 중 세션이 끊겨도 다시 실행하면 이어서 진행"""
    cdir = PRED_DIR / "cache" / RUN_NAME
    cdir.mkdir(parents=True, exist_ok=True)
    parts = []
    for s in range(0, len(df), chunk):
        sub = df.iloc[s:s + chunk]
        ids_h = hashlib.md5("|".join(sub["id"]).encode()).hexdigest()[:8]
        f = cdir / f"{tag}_{MODEL_FINGERPRINT}_tta{n_tta}_t{tokens}_{s:05d}_{ids_h}.npy"
        if f.exists():
            parts.append(np.load(f))
            continue
        lp = predict_logprobs(m, sub, n_tta, tokens, bs, desc=f"{tag} {s}/{len(df)}")
        np.save(f, lp)
        parts.append(lp)
    return np.concatenate(parts) if parts else np.zeros((0, 4), np.float32)

def hires_mask(df, base_lp, thr, hard):
    m = np.exp(base_lp).max(1) < thr
    if hard:
        m |= df["qtype"].isin(HARD_QTYPES).values
    return m

def hires_logprobs(m, df, base_lp, tag, thr, hard):
    """재추론 대상(확신도 < thr, hard면 관계형 유형 포함)만 고해상도(HIRES_TOKENS)로 다시 추론. 나머지는 NaN."""
    hi = np.full_like(base_lp, np.nan)
    sel = np.where(hires_mask(df, base_lp, thr, hard))[0]
    if len(sel) and HIRES_TOKENS > INFER_TOKENS:
        hi[sel] = predict_cached(m, df.iloc[sel].reset_index(drop=True), f"{tag}_hires", N_TTA, HIRES_TOKENS,
                                 bs=max(1, INFER_BS // 2))
    return hi

def apply_hires(base_lp, hi, df, thr, hard):
    out = base_lp.copy()
    sel = np.where(hires_mask(df, base_lp, thr, hard) & ~np.isnan(hi[:, 0]))[0]
    out[sel] = log_softmax_np((base_lp[sel] + hi[sel]) / 2)
    return out, sel

In [ ]:
# 7. 학습 함수 (Drive 체크포인트 · 자동 재개) ───────────────────────────
def load_adapter_weights(m, src):
    sd = load_safetensors(str(Path(src) / "adapter_model.safetensors"))
    res = set_peft_model_state_dict(m, sd)
    unexpected = list(getattr(res, "unexpected_keys", []) or [])
    assert not unexpected, f"어댑터 키 불일치: {unexpected[:3]}"
    print("adapter loaded ←", src)

def _atomic_save(dst, write_fn):
    dst = Path(dst)
    tmp = dst.with_name(dst.name + "_tmp")
    if tmp.exists():
        shutil.rmtree(tmp)
    tmp.mkdir(parents=True)
    write_fn(tmp)
    if dst.exists():
        shutil.rmtree(dst)
    tmp.rename(dst)

def save_ckpt(m, opt, sched, state):
    def _w(d):
        m.save_pretrained(d)
        torch.save({"opt": opt.state_dict(), "sched": sched.state_dict(), "state": state}, d / "trainer_state.pt")
    _atomic_save(RUN_DIR / "last", _w)

def find_ckpt(name):
    for d in (RUN_DIR / name, RUN_DIR / f"{name}_tmp"):
        if (d / "adapter_model.safetensors").exists():
            return d
    return None

def train_model(m, fit_df, val_df):
    trainable = [p for p in m.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(trainable, lr=LR, weight_decay=0.0)
    steps_per_epoch = math.ceil(len(fit_df) / (TRAIN_BS * GRAD_ACCUM))
    total = steps_per_epoch * EPOCHS
    sched = get_cosine_schedule_with_warmup(opt, max(1, int(total * WARMUP_RATIO)), total)
    state = {"epoch": 0, "step_in_epoch": 0, "global_step": 0, "best_acc": -1.0, "history": []}

    ck = find_ckpt("last")
    if ck is not None and (ck / "trainer_state.pt").exists():
        load_adapter_weights(m, ck)
        blob = torch.load(ck / "trainer_state.pt", map_location="cpu", weights_only=False)
        opt.load_state_dict(blob["opt"]); sched.load_state_dict(blob["sched"]); state = blob["state"]
        print(f"▶ 재개: epoch {state['epoch'] + 1}, step {state['step_in_epoch']}/{steps_per_epoch}")

    m.train()
    for epoch in range(state["epoch"], EPOCHS):
        ds = VQADataset(fit_df, train=True, epoch=epoch)
        order = torch.randperm(len(ds), generator=torch.Generator().manual_seed(SEED * 1000 + epoch)).tolist()
        order = order[state["step_in_epoch"] * TRAIN_BS * GRAD_ACCUM:]
        loader = make_loader(ds, TRAIN_BS, sampler=order)
        n_micro, micro, n_oom = len(loader), 0, 0
        recent_loss, recent_acc = [], []
        pbar = tqdm(loader, desc=f"epoch {epoch + 1}/{EPOCHS}")
        for batch in pbar:
            micro += 1
            try:
                logits = letter_logits(m, batch["inputs"])
                tgt = batch["target"].to(DEVICE)
                loss = F.cross_entropy(logits, tgt, label_smoothing=LABEL_SMOOTHING)
                (loss / GRAD_ACCUM).backward()
            except torch.cuda.OutOfMemoryError:
                n_oom += 1
                opt.zero_grad(set_to_none=True)
                torch.cuda.empty_cache()
                if n_oom >= 10 and n_oom > 0.05 * micro:
                    raise RuntimeError("OOM이 반복됩니다 → TRAIN_BS를 줄이거나 TRAIN_TOKENS를 낮추고 GRAD_CKPT=True로 재실행 "
                                       "(체크포인트부터 자동 재개됩니다)")
                continue
            recent_loss = (recent_loss + [loss.item()])[-50:]
            recent_acc = (recent_acc + [(logits.argmax(1) == tgt).float().mean().item()])[-50:]
            if micro % GRAD_ACCUM == 0 or micro == n_micro:
                torch.nn.utils.clip_grad_norm_(trainable, 1.0)
                opt.step(); sched.step(); opt.zero_grad(set_to_none=True)
                state["step_in_epoch"] += 1
                state["global_step"] += 1
                if state["global_step"] % SAVE_EVERY == 0:
                    save_ckpt(m, opt, sched, state)
            pbar.set_postfix(loss=f"{np.mean(recent_loss):.4f}", acc=f"{np.mean(recent_acc):.3f}",
                             lr=f"{sched.get_last_lr()[0]:.1e}", oom=n_oom)

        state["epoch"], state["step_in_epoch"] = epoch + 1, 0
        if val_df is not None and len(val_df):
            vlp = predict_logprobs(m, val_df, n_tta=1, tokens=INFER_TOKENS, desc="valid")
            acc = acc_of(vlp, val_df)
            state["history"].append({"epoch": epoch + 1, "valid_acc": acc})
            print(f"[epoch {epoch + 1}] valid acc (TTA 없이) = {acc:.4f}")
            if acc > state["best_acc"]:
                state["best_acc"] = acc
                _atomic_save(RUN_DIR / "best", m.save_pretrained)
                print("  ↳ best 갱신")
        save_ckpt(m, opt, sched, state)
    (RUN_DIR / "TRAIN_DONE.json").write_text(json.dumps(state, indent=1))
    return state

In [ ]:
# 8. [실행] 모델 로드 + 동작 확인 ────────────────────────────────────────
MODEL_ID = MODEL_REGISTRY[MODEL_KEY]["id"]
processor = setup_processor(MODEL_ID)
model = load_base_model(MODEL_ID)
SUPPORTS_LTK = "logits_to_keep" in inspect.signature(type(model).forward).parameters
TRAIN_DONE = (RUN_DIR / "TRAIN_DONE.json").exists()
if RUN_MODE in ("holdout", "full"):
    model = attach_lora(model)
if RUN_MODE == "tts":        # thinking 켜기 (템플릿이 지원할 때만)
    CHAT_KWARGS = {"enable_thinking": True} if CHAT_KWARGS else {}
import transformers, peft, accelerate
ENV_INFO = {"model_id": MODEL_ID, "model_revision": MODEL_REVISION,
            "model_commit": getattr(model.config, "_commit_hash", None), "gpu": GPU_NAME, "use_4bit": USE_4BIT,
            "torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__,
            "accelerate": accelerate.__version__, "seed": SEED, "split_seed": SPLIT_SEED,
            "config": {k: globals()[k] for k in ["EPOCHS", "LR", "LORA_R", "LORA_VISION", "LABEL_SMOOTHING",
                       "TRAIN_TOKENS", "INFER_TOKENS", "HIRES_TOKENS", "N_TTA", "TRAIN_BS", "GRAD_ACCUM", "USE_PSEUDO"]}}
(RUN_DIR / "env.json").write_text(json.dumps(ENV_INFO, indent=1, default=str))
print(f"PATCH_FACTOR={PATCH_FACTOR} | LETTER_IDS={LETTER_IDS} | logits_to_keep={SUPPORTS_LTK} | "
      f"chat_kwargs={CHAT_KWARGS} | train_done={TRAIN_DONE}")

# 생성 결과가 실제로 알파벳 한 글자인지 눈으로 확인
model.eval()
for _, r in valid_df.head(2 if RUN_MODE != "tts" else 0).iterrows():
    enc = processor(text=[render_prompt(r["question"], [r[c] for c in OPT_COLS])],
                    images=[load_image(r["abs_path"], INFER_TOKENS)], return_tensors="pt").to(DEVICE)
    with torch.no_grad(), torch.autocast(device_type=DEVICE, dtype=torch.bfloat16):
        gen = model.generate(**enc, max_new_tokens=6, do_sample=False, use_cache=True)
    print(f"Q: {r['question'][:50]} | gold={r['answer'].upper()} | 생성={processor.batch_decode(gen[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)[0]!r}")

if QUICK_CHECK_N and not TRAIN_DONE and len(valid_df) and RUN_MODE != "tts":
    _q = valid_df.head(QUICK_CHECK_N)
    t0 = time.time()
    _acc = acc_of(predict_logprobs(model, _q, n_tta=1), _q)
    spp = (time.time() - t0) / len(_q)
    print(f"zero-shot valid acc (앞 {len(_q)}개, TTA 없음) = {_acc:.4f} | {spp:.2f}s/샘플")
    if RUN_MODE != "zeroshot":
        print(f"  예상 학습 ≈ {spp * 3 * len(train_fit_df) * EPOCHS / 3600:.1f}h (대략)", end=" | ")
    print(f"예상 추론(valid+test, TTA{N_TTA}) ≈ {spp * N_TTA * (len(valid_df) + len(test_df)) * 1.15 / 3600:.1f}h (대략)")

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/1058 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

trainable params: 268,435,456 || all params: 33,625,825,520 || trainable%: 0.7983
PATCH_FACTOR=32 | LETTER_IDS=[32, 33, 34, 35] | logits_to_keep=True | chat_kwargs={} | train_done=False
Q: 2026 NEW CONTENT ACADEMY에서 주최하는 쇼케이스 행사는 언제 열리나요? | gold=B | 생성='B'
Q: 사진 속 조형물 받침대에 적혀 있는 글자는 무엇인가요? | gold=C | 생성='C'


infer:   0%|          | 0/100 [00:00<?, ?it/s]

zero-shot valid acc (앞 200개, TTA 없음) = 0.9150 | 0.36s/샘플
  예상 학습 ≈ 4.0h (대략) | 예상 추론(valid+test, TTA4) ≈ 3.4h (대략)


In [ ]:
# 9. [실행] 학습 ─────────────────────────────────────────────────────────
if RUN_MODE in ("holdout", "full") and not TRAIN_DONE:
    t0 = time.time()
    st = train_model(model, train_fit_df, valid_df if RUN_MODE == "holdout" else None)
    print(f"학습 완료: {(time.time() - t0) / 60:.1f}분 | history={st['history']}")
    TRAIN_DONE = True
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
else:
    print("학습 생략 (zeroshot/tts 또는 이미 완료된 run)")

epoch 1/2:   0%|          | 0/3357 [00:40<?, ?it/s]

epoch 2/2:   0%|          | 0/3357 [00:00<?, ?it/s]

학습 완료: 287.0분 | history=[]


In [ ]:
# 10. [실행] 추론 (순열 TTA + 고해상도 재추론) · 예측 저장 · 단일모델 제출 ──
if RUN_MODE == "tts":
    print("tts 모드: 개별 추론 생략 → 11번 앙상블 후 13번 셀에서 재판단")
else:
    if RUN_MODE != "zeroshot":
        src = (find_ckpt("best") if RUN_MODE == "holdout" else None) or find_ckpt("last")
        assert src is not None, "학습된 어댑터가 없습니다. 학습 셀을 먼저 실행하세요."
        _st = (Path(src) / "adapter_model.safetensors").stat()
        MODEL_FINGERPRINT = hashlib.md5(f"{_st.st_size}_{_st.st_mtime}_{SYSTEM_PROMPT}".encode()).hexdigest()[:8]
        if hasattr(model, "peft_config"):         # (이 셀을 다시 실행해도 안전하도록: 이미 병합됐으면 건너뜀)
            load_adapter_weights(model, src)
            if not USE_4BIT:
                model = model.merge_and_unload()  # LoRA 병합 → 추론 10~15% 빠름
    else:
        MODEL_FINGERPRINT = hashlib.md5(SYSTEM_PROMPT.encode()).hexdigest()[:8]
    model.eval()

    meta = {"run_name": RUN_NAME, "model_key": MODEL_KEY, "mode": RUN_MODE, "pseudo": bool(USE_PSEUDO),
            "seed": SEED, "tag": TAG,
            "smoke": SMOKE_TEST, "n_tta": N_TTA, "time": time.time(), "has_valid": False, "valid_acc": None}
    has_valid = RUN_MODE in ("holdout", "zeroshot") and len(valid_df) > 0
    if has_valid:
        v_base = predict_cached(model, valid_df, "valid")
        v_hi = hires_logprobs(model, valid_df, v_base, "valid", max(HIRES_GRID), True)   # 모든 후보의 합집합
        a_base = acc_of(v_base, valid_df)
        HIRES_T, HIRES_HARD, best_acc = 0.0, False, a_base
        print(f"valid acc: TTA{N_TTA} = {a_base:.4f}")
        for hard in (False, True):                                   # 비용이 싼 조합부터 → 동점이면 싼 쪽 유지
            for t in (0.0,) + tuple(HIRES_GRID):
                if t == 0.0 and not hard:
                    continue
                _lp, _sel = apply_hires(v_base, v_hi, valid_df, t, hard)
                _a = acc_of(_lp, valid_df)
                print(f"  + 고해상도 (conf<{t}{' + 관계형 유형' if hard else ''}: {len(_sel)}개) = {_a:.4f}")
                if _a > best_acc + 1e-9:
                    HIRES_T, HIRES_HARD, best_acc = t, hard, _a
        v_final = apply_hires(v_base, v_hi, valid_df, HIRES_T, HIRES_HARD)[0]
        print(f"→ 선택: 임계값 {HIRES_T}, 관계형 유형 포함 {HIRES_HARD} (둘 다 0/False = 사용 안 함)")
        _va = valid_df.assign(correct=(v_final.argmax(1) == valid_df["answer"].map(OPT_COLS.index).values))
        print(_va.groupby("qtype")["correct"].agg(["mean", "count"]).sort_values("mean").round(4))
        np.save(PRED_DIR / f"{RUN_NAME}__valid.npy", v_final)
        meta.update(has_valid=True, valid_acc=acc_of(v_final, valid_df), valid_acc_base=a_base,
                    valid_ids_md5=hashlib.md5("|".join(valid_df["id"]).encode()).hexdigest())
    else:
        prev = [json.loads(f.read_text()) for f in PRED_DIR.glob(f"{MODEL_KEY}_holdout_*__meta.json")]
        prev = [m for m in prev if m.get("tag", "") == TAG and not m.get("smoke")]
        _pm = max(prev, key=lambda m: m["time"]) if prev else {}
        HIRES_T, HIRES_HARD = _pm.get("hires_t", 0.9), _pm.get("hires_hard", False)

    t_base = predict_cached(model, test_df, "test")
    if HIRES_T or HIRES_HARD:
        t_final = apply_hires(t_base, hires_logprobs(model, test_df, t_base, "test", HIRES_T, HIRES_HARD),
                              test_df, HIRES_T, HIRES_HARD)[0]
    else:
        t_final = t_base
    np.save(PRED_DIR / f"{RUN_NAME}__test.npy", t_final)
    if len(dev_df) and PREDICT_DEV:                   # dev 라벨 생성용 (학습에 dev를 쓴 run의 dev 예측은 앙상블이 사용하지 않음)
        d_base = predict_cached(model, dev_df, "dev")
        if HIRES_T or HIRES_HARD:
            d_base = apply_hires(d_base, hires_logprobs(model, dev_df, d_base, "dev", HIRES_T, HIRES_HARD),
                                 dev_df, HIRES_T, HIRES_HARD)[0]
        np.save(PRED_DIR / f"{RUN_NAME}__dev.npy", d_base)
        meta.update(has_dev=True, dev_ids_md5=hashlib.md5("|".join(dev_df["id"]).encode()).hexdigest())
    meta["used_dev"] = bool(USE_DEV)
    meta["hires_t"], meta["hires_hard"] = float(HIRES_T), bool(HIRES_HARD)
    (PRED_DIR / f"{RUN_NAME}__meta.json").write_text(json.dumps(meta, indent=1))

    if not SMOKE_TEST:
        sub = sample_sub.copy()
        pred_map = dict(zip(test_df["id"], [OPT_COLS[i] for i in t_final.argmax(1)]))
        sub["answer"] = sub["id"].map(pred_map)
        assert sub["answer"].notna().all()
        sub.to_csv(RUN_DIR / "submission.csv", index=False)
        if IN_COLAB:
            sub.to_csv(f"/content/submission_{RUN_NAME}.csv", index=False)
        print("저장:", RUN_DIR / "submission.csv")
        print(sub["answer"].value_counts(normalize=True).sort_index().round(3).to_dict())
    print(json.dumps(meta, indent=1, ensure_ascii=False))

adapter loaded ← /content/drive/MyDrive/vqa_challenge/runs/qwen3vl_32b_full_s42_p3/last


test 0/6714:   0%|          | 0/500 [00:00<?, ?it/s]

test 1000/6714:   0%|          | 0/500 [00:40<?, ?it/s]

## 앙상블 · pseudo label
저장된 모든 run의 예측(`preds/*__meta.json`)을 모아 **검증셋 NLL 기준으로 가중치를 최적화**하고 최종 제출 파일을 만듭니다.
같은 모델의 `full` run이 있으면 test 예측은 `full` run 것을, 가중치 튜닝은 `holdout` run의 검증 예측을 사용합니다.
이 셀은 모델 로드 없이 단독 실행 가능합니다 (설정 셀 1~3만 먼저 실행).

In [ ]:
# 11. 앙상블 + 최종 제출 + pseudo label 생성 ─────────────────────────────
y_valid = valid_df["answer"].map(OPT_COLS.index).values
valid_md5 = hashlib.md5("|".join(valid_df["id"]).encode()).hexdigest()
metas = [json.loads(f.read_text()) for f in sorted(PRED_DIR.glob("*__meta.json"))]
metas = [m for m in metas if m.get("smoke", False) == SMOKE_TEST]

groups = {}
for m in metas:
    key = f"{m['model_key']}_{'zs' if m['mode'] == 'zeroshot' else 'ft'}"
    if m["mode"] != "zeroshot":
        key += f"_s{m.get('seed', 42)}" + (f"_{m['tag']}" if m.get("tag") else "")
    groups.setdefault(key, []).append(m)

members = []
for g, ms in sorted(groups.items()):
    vc = [m for m in ms if m["has_valid"] and m.get("valid_ids_md5") == valid_md5]
    if not vc:
        print(f"[skip] {g}: 검증 예측 없음")
        continue
    vm = max(vc, key=lambda m: (m["valid_acc"], m["time"]))
    tm = max([m for m in ms if m["mode"] == "full" and m.get("used_dev", False) == vm.get("used_dev", False)] or [vm],
             key=lambda m: (m["pseudo"], m["time"]))
    members.append({"name": g, "group": ms, "valid": np.load(PRED_DIR / f"{vm['run_name']}__valid.npy"),
                    "test": np.load(PRED_DIR / f"{tm['run_name']}__test.npy"),
                    "vrun": vm["run_name"], "trun": tm["run_name"]})
assert members, "앙상블할 예측이 없습니다. 먼저 holdout/zeroshot run을 실행하세요."

def nll_of(lp):
    return float(-lp[np.arange(len(y_valid)), y_valid].mean())

def combine(lps, w):
    return log_softmax_np(sum(wi * lp for wi, lp in zip(w, lps)))

V = [mb["valid"] for mb in members]
T = [mb["test"] for mb in members]
for mb in members:
    print(f"{mb['name']:<24} valid acc={acc_of(mb['valid'], valid_df):.4f} nll={nll_of(mb['valid']):.4f} "
          f"| valid←{mb['vrun']} | test←{mb['trun']}")

w = np.ones(len(members)) / len(members)
print(f"균등 가중 앙상블: acc={acc_of(combine(V, w), valid_df):.4f}")
grid = np.round(np.arange(0, 1.0001, 0.05), 2)
for _ in range(10):                                       # 좌표 하강 (NLL 최소화: 정확도보다 노이즈에 강함)
    changed = False
    for i in range(len(members)):
        best = (nll_of(combine(V, w)), w[i])
        for gv in grid:
            w2 = w.copy(); w2[i] = gv
            if w2.sum() == 0:
                continue
            s = nll_of(combine(V, w2 / w2.sum()))
            if s < best[0] - 1e-6:
                best = (s, gv)
        if best[1] != w[i]:
            w[i] = best[1]; w = w / w.sum(); changed = True
    if not changed:
        break
w_eq = np.ones(len(members)) / len(members)
acc_t, acc_e = acc_of(combine(V, w), valid_df), acc_of(combine(V, w_eq), valid_df)
if acc_t < acc_e - 0.005:                                 # 검증 ~3개 이상 손해면 튜닝 결과를 믿지 않음
    print(f"튜닝 가중치 acc {acc_t:.4f} < 균등 {acc_e:.4f} → 균등 가중치 사용")
    w = w_eq
ens_valid, ens_test = combine(V, w), combine(T, w)

def mcnemar_p(ok_a, ok_b):
    """두 예측의 정오답 쌍 비교 (정확 이항검정, 양측). 작을수록 차이가 우연이 아님."""
    from math import comb
    b, c = int((ok_a & ~ok_b).sum()), int((~ok_a & ok_b).sum())
    n = b + c
    return 1.0 if n == 0 else min(1.0, 2 * sum(comb(n, k) for k in range(min(b, c) + 1)) / 2 ** n)

_ok_ens = ens_valid.argmax(1) == y_valid
for mb in members:
    _ok = mb["valid"].argmax(1) == y_valid
    print(f"  앙상블 vs {mb['name']:<24}: 앙상블만 맞춤 {int((_ok_ens & ~_ok).sum())} / 멤버만 맞춤 {int((~_ok_ens & _ok).sum())}"
          f" | McNemar p={mcnemar_p(_ok_ens, _ok):.3f}")
print("가중치:", {mb["name"]: round(float(x), 3) for mb, x in zip(members, w)},
      f"→ valid acc={acc_of(ens_valid, valid_df):.4f}")

final = sample_sub.copy()
final["answer"] = final["id"].map(dict(zip(test_df["id"], [OPT_COLS[i] for i in ens_test.argmax(1)])))
if not SMOKE_TEST:
    assert final["answer"].notna().all()
    final.to_csv(DRIVE_ROOT / "submission_ensemble.csv", index=False)
    if IN_COLAB:
        final.to_csv("/content/submission.csv", index=False)
    print("최종 제출:", DRIVE_ROOT / "submission_ensemble.csv")
np.save(PRED_DIR / "_ensemble_valid.npy", ens_valid)
np.save(PRED_DIR / "_ensemble_test.npy", ens_test)

v_conf, t_conf = np.exp(ens_valid).max(1), np.exp(ens_test).max(1)
mask_v = v_conf >= PSEUDO_THRESHOLD
print(f"pseudo 기준 {PSEUDO_THRESHOLD}: valid에서 {mask_v.mean():.1%} 해당, 그 정확도 "
      f"{(ens_valid.argmax(1)[mask_v] == y_valid[mask_v]).mean() if mask_v.any() else float('nan'):.4f}")
pl = pd.DataFrame({"id": test_df["id"], "answer": [OPT_COLS[i] for i in ens_test.argmax(1)], "conf": t_conf})
pl = pl[pl["conf"] >= PSEUDO_THRESHOLD]
if not SMOKE_TEST:
    pl.to_csv(PSEUDO_CSV, index=False)
    print(f"pseudo label: test {len(pl)}/{len(test_df)}개 → {PSEUDO_CSV}")

# dev 라벨 생성: 모델 예측(dev를 학습에 쓰지 않은 run만) + 교육생 투표 ─────────────
# 투표 패턴(3표 단독 / 2표 단독 / 동률)마다, '모델이 확신한 문항'에서 다수결과 모델의 일치율을 잰다.
# 일치율 ≥ DEV_TRUST인 패턴은 다수결을 라벨로 신뢰(모델이 불확실한 어려운 문항까지 포함 → 새 정보),
# 아닌 패턴은 모델이 확신하고 투표와도 모순되지 않는 문항만 사용.
if len(dev_df):
    d_md5 = hashlib.md5("|".join(dev_df["id"]).encode()).hexdigest()
    D, Dw = [], []
    for mb, wi in zip(members, w):             # 같은 모델 그룹에서 dev를 학습에 쓰지 않은 run의 dev 예측 (최신)
        ok = [m for m in mb["group"] if m.get("has_dev") and not m.get("used_dev") and m.get("dev_ids_md5") == d_md5]
        if ok:
            D.append(np.load(PRED_DIR / f"{max(ok, key=lambda m: m['time'])['run_name']}__dev.npy")); Dw.append(wi)
    if not D:
        print("dev 예측이 있는 run이 없습니다 → dev를 쓰려면 학습된 run의 전체 셀을 한 번 더 실행하세요 (학습·test는 캐시 사용).")
    else:
        Dw = np.array(Dw) if sum(Dw) > 0 else np.ones(len(D))
        ens_dev = combine(D, Dw / Dw.sum())
        pm = np.exp(ens_dev); conf, am = pm.max(1), pm.argmax(1)
        R = dev_df[DEV_COLS].astype(str).apply(lambda c: c.str.strip().str.lower())
        cnt = np.stack([(R == o).sum(1).values for o in OPT_COLS], 1)
        top = cnt.max(1); uniq = (cnt == top[:, None]).sum(1) == 1; maj = cnt.argmax(1)
        bucket = np.where(uniq & (top >= 3), "3표 단독", np.where(uniq & (top == 2), "2표 단독", "동률/무응답"))
        lab, src = np.full(len(dev_df), -1), np.array([""] * len(dev_df), dtype=object)
        print(f"dev 라벨링 (모델 {len(D)}개 앙상블, 확신 기준 {PSEUDO_THRESHOLD}, 신뢰 기준 {DEV_TRUST})")
        for b in ["3표 단독", "2표 단독", "동률/무응답"]:
            sel = bucket == b
            cs = sel & (conf >= PSEUDO_THRESHOLD)
            agree = float((am[cs] == maj[cs]).mean()) if cs.any() and b != "동률/무응답" else float("nan")
            trust = b != "동률/무응답" and agree >= DEV_TRUST
            if trust:        # 다수결 신뢰. 단, 모델이 매우 확신하며 반대하는 문항은 애매하므로 제외
                use = sel & ~((conf >= 0.99) & (am != maj))
                lab[use], src[use] = maj[use], "다수결"
            else:            # 모델 확신 + 모델 답이 최다 득표 중 하나
                use = cs & (cnt[np.arange(len(cnt)), am] == top)
                lab[use], src[use] = am[use], "모델"
            n_new = int((use & (am != lab) & (lab >= 0)).sum())
            print(f"  {b:<7} {int(sel.sum()):5d}개 | 모델확신 문항의 다수결 일치율 {agree:.3f} | "
                  f"{'다수결 신뢰' if trust else '모델 확신만'} → 라벨 {int(use.sum())}개 (모델 예측과 다른 라벨 {n_new}개)")
        keep = lab >= 0
        dl = pd.DataFrame({"id": dev_df["id"][keep], "answer": [OPT_COLS[i] for i in lab[keep]], "source": src[keep]})
        if not SMOKE_TEST:
            dl.to_csv(DEV_LABEL_CSV, index=False)
        print(f"→ dev 라벨 {keep.sum()}/{len(dev_df)}개 {'저장: ' + str(DEV_LABEL_CSV) if not SMOKE_TEST else '(smoke: 저장 안 함)'}")

In [ ]:
# 12. 오류 분석 ──────────────────────────────────────────────────────────
va = valid_df.copy()
va["pred"] = [OPT_COLS[i] for i in ens_valid.argmax(1)]
va["conf"] = np.exp(ens_valid).max(1)
va["correct"] = va["pred"] == va["answer"]
print(va.groupby("qtype")["correct"].agg(["mean", "count"]).sort_values("mean").round(4))
va["conf_bin"] = pd.cut(va["conf"], [0, 0.5, 0.7, 0.9, 0.97, 1.0001])
print(va.groupby("conf_bin", observed=True)["correct"].agg(["mean", "count"]).round(4))

wrong = va[~va["correct"]].sort_values("conf", ascending=False)
print(f"\n오답 {len(wrong)}개 — 확신하고 틀린 순 (라벨 노이즈 / 작은 글씨 / 회전 여부 확인)")
for _, r in wrong.head(15).iterrows():
    print(f"[{r['id']}] ({r['qtype']}) conf={r['conf']:.2f} gold={r['answer']}:{r[r['answer']]} | "
          f"pred={r['pred']}:{r[r['pred']]}\n    Q: {r['question']}")

if IN_COLAB and len(wrong):
    from IPython.display import display
    for _, r in wrong.head(4).iterrows():
        im = load_image(r["abs_path"], 600)
        print(r["id"], r["question"])
        display(im)

## 선택적 추론 재판단 (test-time scaling) — `RUN_MODE = "tts"`
앙상블이 헷갈리는 문항(확신도 낮음)에만 thinking 모델이 단계별로 추론하게 한 뒤, **추론을 닫고 A~D 토큰 확률로 채점**합니다
(문자열 파싱 없음 → 답을 못 내는 실패 0). 적용 여부·임계값·가중치는 **검증셋 5-fold 교차검증**으로 정하고,
교차검증 이득이 없으면 앙상블 결과를 그대로 씁니다. 결과는 문항 id별로 Drive에 캐시되어, 앙상블이 바뀌어도 새 문항만 추가 추론합니다.
최종 제출 파일: `vqa_challenge/submission_final.csv`

In [ ]:
# 13. [RUN_MODE="tts"] 선택적 추론 재판단 ──────────────────────────────
if RUN_MODE == "tts":
    TTS_CACHE = PRED_DIR / f"tts_{MODEL_KEY}_{PROMPT_VER}.json"
    tts_cache = json.loads(TTS_CACHE.read_text()) if TTS_CACHE.exists() else {}
    v_conf, t_conf = np.exp(ens_valid).max(1), np.exp(ens_test).max(1)
    print(f"재판단 후보: valid {int((v_conf < TTS_CONF_MAX).sum())}개 → 먼저 검증셋으로 효과를 판정하고, "
          f"채택될 때만 test의 해당 문항을 추론합니다")

    END_THINK = "</think>"
    def _clean(t):
        for tok in (processor.tokenizer.eos_token, processor.tokenizer.pad_token, "<|im_end|>", "<|endoftext|>"):
            if tok:
                t = t.replace(tok, "")
        return t

    model.eval()
    torch.manual_seed(SEED)

    def _think_rows(rows):
        """rows를 thinking으로 풀고 캐시에 저장. 메모리 부족이면 절반씩 나눠 재시도."""
        try:
            _think_batch(rows)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if len(rows) == 1:
                raise
            h = len(rows) // 2
            _think_rows(rows.iloc[:h]); _think_rows(rows.iloc[h:])

    def run_tts(df, desc):
        todo = df[~df["id"].isin(tts_cache)].reset_index(drop=True)
        print(f"{desc}: {len(df)}개 중 새로 추론 {len(todo)}개 (나머지는 캐시)")
        for s0 in tqdm(range(0, len(todo), TTS_BS), desc=desc):
            _think_rows(todo.iloc[s0:s0 + TTS_BS])
            TTS_CACHE.write_text(json.dumps(tts_cache))     # 배치마다 저장 → 끊겨도 이어서
        if len(todo):
            print(f"  추론을 스스로 마친 비율: {np.mean([tts_cache[i]['closed'] for i in todo['id']]):.1%}")

    def _think_batch(rows):
        imgs = [load_image(p, INFER_TOKENS) for p in rows["abs_path"]]
        prompts = [render_prompt(r["question"], [r[c] for c in OPT_COLS]) for _, r in rows.iterrows()]
        enc = processor(text=prompts, images=imgs, padding=True, return_tensors="pt").to(DEVICE)
        with torch.no_grad(), torch.autocast(device_type=DEVICE, dtype=torch.bfloat16):
            gen = model.generate(**enc, max_new_tokens=TTS_MAX_TOKENS, do_sample=True, temperature=1.0,
                                 top_p=0.95, top_k=20, use_cache=True)
        outs = processor.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=False)
        score_texts, closed = [], []
        for pr, o in zip(prompts, outs):
            o = _clean(o)
            if END_THINK in o:
                reasoning, c = o.split(END_THINK, 1)[0], True
            else:
                reasoning, c = o, False              # 길이 초과 → 강제로 닫음 (budget forcing)
            score_texts.append(pr + reasoning.rstrip() + "\n" + END_THINK + "\n\n")
            closed.append(c)
        enc2 = processor(text=score_texts, images=imgs, padding=True, return_tensors="pt")
        with torch.no_grad():
            lp = letter_logits(model, dict(enc2)).log_softmax(-1).cpu().numpy()
        for (_, r), l, c in zip(rows.iterrows(), lp, closed):
            tts_cache[r["id"]] = {"lp": [float(x) for x in l], "closed": c}

    run_tts(valid_df[v_conf < TTS_CONF_MAX], "thinking(valid)")

    def _tts_matrix(df):
        return np.array([tts_cache[i]["lp"] if i in tts_cache else [np.nan] * 4 for i in df["id"]], dtype=np.float32)
    H_v = _tts_matrix(valid_df)

    def apply_tts(E, H, tau, alpha):
        out = E.copy()
        m = (np.exp(E).max(1) < tau) & ~np.isnan(H[:, 0])
        out[m] = log_softmax_np((1 - alpha) * E[m] + alpha * H[m])
        return out

    GRID = [(t, a) for t in (0.6, 0.8, 0.9, 0.95, 0.97, 0.99) if t <= TTS_CONF_MAX for a in (0.25, 0.5, 0.75, 1.0)]
    def pick(idx):                                  # idx 안에서 최고 (동점이면 더 보수적인 설정), 기준선보다 나아야 채택
        base = int((ens_valid[idx].argmax(1) == y_valid[idx]).sum())
        best, best_n = None, base
        for t, a in sorted(GRID, key=lambda x: (x[1], x[0])):
            n = int((apply_tts(ens_valid[idx], H_v[idx], t, a).argmax(1) == y_valid[idx]).sum())
            if n > best_n:
                best, best_n = (t, a), n
        return best

    # 5-fold 교차검증: 설정을 4개 fold에서 고르고 남은 fold에서 평가 → 과적합 없는 이득 추정
    rng = np.random.default_rng(SEED)
    folds = rng.permutation(len(valid_df)) % 5
    cv_ok = np.zeros(len(valid_df), dtype=bool)
    for k in range(5):
        tr_i, te_i = np.where(folds != k)[0], np.where(folds == k)[0]
        prm = pick(tr_i)
        pred = ens_valid[te_i] if prm is None else apply_tts(ens_valid[te_i], H_v[te_i], *prm)
        cv_ok[te_i] = pred.argmax(1) == y_valid[te_i]
    base_ok = ens_valid.argmax(1) == y_valid
    gain = int(cv_ok.sum()) - int(base_ok.sum())
    alone = ~np.isnan(H_v[:, 0])
    print(f"thinking 단독 정확도 (재판단 후보 {int(alone.sum())}개): "
          f"{(H_v[alone].argmax(1) == y_valid[alone]).mean():.3f} vs 앙상블 {(base_ok[alone]).mean():.3f}")
    print(f"교차검증: 앙상블 {base_ok.mean():.4f} → 재판단 {cv_ok.mean():.4f} ({gain:+d}문항) "
          f"| McNemar p={mcnemar_p(cv_ok, base_ok):.3f}")

    final_prm = pick(np.arange(len(valid_df))) if gain >= 2 else None
    if final_prm:
        print(f"→ 검증 판정: 채택 (확신도 < {final_prm[0]}, thinking 가중치 {final_prm[1]}) "
              f"→ test 중 해당 {int((t_conf < final_prm[0]).sum())}개만 추론합니다")
        run_tts(test_df[t_conf < final_prm[0]], "thinking(test)")
        H_t = _tts_matrix(test_df)
        t_out = apply_tts(ens_test, H_t, *final_prm)
        v_out = apply_tts(ens_valid, H_v, *final_prm)
        print(f"→ 채택: 확신도 < {final_prm[0]} 문항에 thinking 가중치 {final_prm[1]} "
              f"| 전체 검증 {acc_of(v_out, valid_df):.4f} | test 변경 {int((t_out.argmax(1) != ens_test.argmax(1)).sum())}문항")
    else:
        t_out = ens_test
        print("→ 교차검증 이득이 2문항 미만 → 채택하지 않음 (앙상블 결과 그대로 사용)")
    final = sample_sub.copy()
    final["answer"] = final["id"].map(dict(zip(test_df["id"], [OPT_COLS[i] for i in t_out.argmax(1)])))
    if not SMOKE_TEST:
        assert final["answer"].notna().all()
        final.to_csv(DRIVE_ROOT / "submission_final.csv", index=False)
        print("최종 제출:", DRIVE_ROOT / "submission_final.csv")